In [8]:
from __future__ import annotations
import os
import pandas as pd
from spread_adf_kss import (
    kss_table_for_alts,
    read_copula_top2,
    result_adf_kss_dir,
    run_full_pipeline,
)

TF = "1h"

## 1. symbol selection

we have the data:
- $B_t$: `BTC_USDT` close
- $P_t^{(i)}$: close of altcoin $i$

For each altcoin $i$, fit

$$
B_t = \beta_i P_t^{(i)} + \varepsilon_t^{(i)}
$$

Then spread (i.e. residual)

$$
S_t^{(i)} = B_t - \hat\beta_i P_t^{(i)} = \hat\varepsilon_t^{(i)}.
$$

For each spread, run the KSS-style Taylor auxiliary regression:

$$
\Delta S_t^{(i)} = \delta_i\big(S_{t-1}^{(i)}\big)^3 + u_t.
$$

$$
\text{pass KSS} \iff t(\hat\delta_i) < -1.92. \text{(At 10\% level)}
$$



Then for each pair $(BTC, i)$, compute Kendall rank correlation on aligned closes:

$$
\tau_i = \tau(B_t, P_t^{(i)}).
$$

Kendall correlation measures rank-consistency (monotonic dependence), and we prefer it over Pearson because crypto returns are often non-Gaussian with outliers/nonlinearity, where linear correlation can be unstable or misleading.

Current selection rule is:
1. keep only symbols with `kss_pass_10pct = True`;
2. rank by `kendall_tau` descending, and select top 2 symbols as Copula candidates.

In [9]:
summary_path = os.path.join(result_adf_kss_dir(TF), "spread_adf_kss_summary.csv")
cols = [
    "alt_symbol",
    "kendall_tau",
    "adf_statistic",
    "adf_pvalue",
    "adf_used_lags",
    "adf_pass_5pct",
    "kss_t_stat",
    "kss_pass_10pct",
]
display(pd.read_csv(summary_path)[cols])

,alt_symbol,kendall_tau,adf_statistic,adf_pvalue,adf_used_lags,adf_pass_5pct,kss_t_stat,kss_pass_10pct
0,ADA_USDT,0.291216,-0.819511,0.813323,53,False,-2.290940,True
1,BCH_USDT,0.555962,-2.239289,0.192296,54,False,-5.953640,True
2,ETC_USDT,0.003135,-1.857445,0.352337,55,False,-6.797596,True
3,ETH_USDT,0.586205,-1.334392,0.613295,52,False,-1.494605,False
4,LINK_USDT,0.427718,-1.505115,0.530980,54,False,-4.045046,True
5,LTC_USDT,0.284595,-1.328941,0.615830,53,False,-3.748960,True
6,XRP_USDT,0.624554,-2.771797,0.062417,55,False,-2.397124,True


We will run copula model on XRP/BCH.

In [ ]:
from spread_adf_kss import discover_csv_paths, load_close_series, merge_on_timestamp

paths = discover_csv_paths("1h")
pair_hour_df = merge_on_timestamp(
    {
        "XRP_USDT": load_close_series(paths["XRP_USDT"]),
        "BCH_USDT": load_close_series(paths["BCH_USDT"]),
    }
).reset_index().rename(columns={"index": "timestamp"})

,timestamp,XRP_USDT,BCH_USDT
0,1609459200000,0.2217,341.32
1,1609462800000,0.2238,349.88
2,1609466400000,0.2238,354.32
3,1609470000000,0.2273,350.81
4,1609473600000,0.2393,347.92


# 2. Copula
- fit marginals first, then apply tranformation to get $(u_t, v_t) \in (0,1)^2$;
- fit copulas (Gaussian, Student-t, Clayton, Gumbel, Frank) via MLE, select the best by AIC;
- computational efficiency of Nelder-Mead vs BFGS.

# 3. Mispricing Index / Trading signal
- compute MI from copula conditional CDFs (e.g., $\partial C(u_1,u_2)/\partial u_2$ and $\partial C(u_1,u_2)/\partial u_1$), then center as $MI=\text{conditional prob}-0.5$;
- generate entry/exit signals by MI thresholds, backtest and display signals on the price plot.